In [7]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt

In [21]:
df = pd.read_csv('compl_enum_cstr_5_baron_hull.csv')

def load_grid(filename="compl_enum_cstr_5_baron_hull.csv"):
    """
    Load grid data from CSV.
    Expected CSV columns: "x", "y", "Objective", among others.
    Returns a dictionary mapping (x, y) tuples to the objective value.
    """
    grid_data = {}
    with open(filename, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                # Convert x and y values to integers
                x_val = int(row["x"])
                y_val = int(row["y"])
                # Convert objective value to float
                obj_val = float(row["Objective"])
                grid_data[(x_val, y_val)] = obj_val
            except Exception as e:
                # Skip rows with conversion issues
                continue

    # Compute the minimum among finite objective values
    finite_values = [v for v in grid_data.values() if not np.isnan(v)]
    x_min = min(finite_values) if finite_values else None         
    return grid_data, x_min
# Load the data from grid.csv
grid_data, x_min = load_grid()

# Determine grid bounds from the keys of grid_data
xs = [pt[0] for pt in grid_data.keys()]
ys = [pt[1] for pt in grid_data.keys()]
min_x, max_x = min(xs), max(xs)
min_y, max_y = min(ys), max(ys)
print("Grid bounds:")
print("x in [{}, {}]".format(min_x, max_x))
print("y in [{}, {}]".format(min_y, max_y))

Grid bounds:
x in [1, 5]
y in [1, 5]


In [22]:
# Load the data from grid.csv
grid_data, x_min = load_grid()

# PENALTY = 1e6  # adjust this penalty as appropriate
PENALTY = np.inf # use infinity as penalty for infeasible points
# PENALTY = np.nan  # use NaN as penalty for infeasible points
# PENALTY = None  # use None as penalty for infeasible points

def obj_funct(x):
    """
    Given a candidate point x (a list or 1-D numpy array with 2 elements),
    this function rounds x to the nearest integers and returns the objective value
    if the point exists in grid_data and is feasible.
    Otherwise, returns a penalty value.
    """
    # Round the candidate coordinates to nearest integers
    x_int = int(round(x[0]))
    y_int = int(round(x[1]))
    
    # Check that x_int and y_int are numbers (optional check)
    if not isinstance(x_int, (int, float)) or not isinstance(y_int, (int, float)):
        return PENALTY
    
    # If the point exists in the grid, check its objective value.
    if (x_int, y_int) in grid_data:
        obj_val = grid_data[(x_int, y_int)]
        # Return penalty if the objective is NaN.
        if np.isnan(obj_val):
            return PENALTY
        else:
            return obj_val
    else:
        # If the point is not in the grid, return the penalty.
        return PENALTY

In [23]:
# Test for the obj_funct function
obj_funct([7,5])

inf

$$
f(x)=-1+\frac{4}{\pi} \arctan(\frac{x-x_{min}}{\alpha})
$$

In [ ]:
def scale_point(x, y, grid_data, x_min, alpha=1.0):
    """
    Compute the arctan-based scaled transformation for a point (x, y).
    Maps x_min -> -1 and infinity -> 1.
    
    Parameters:
      - x, y: coordinates (floats)
      - grid_data: dict from load_grid
      - x_min: minimum objective value from load_grid
      - alpha: scale parameter (default=1)
    
    Returns:
      - scaled value in [-1, 1]
    """
    raw_val = obj_funct([x, y])
    delta = raw_val - x_min
    scaled = -1 + (4 / np.pi) * np.arctan(delta / alpha)
    return float(scaled)

# scale_point(1, 1, grid_data, x_min, alpha=1.0)

for i in range(1,6):
    for j in range(1,6):
        x = i
        y = j
        print(f"({x}, {y}) -> {scale_point(x, y, grid_data, x_min, alpha=1.0)}")

(1, 1) -> 0.8147049262704746
(1, 2) -> 1.0
(1, 3) -> 1.0
(1, 4) -> 1.0
(1, 5) -> 1.0
(2, 1) -> -8.46073986315865e-05
(2, 2) -> -8.460721999759002e-05
(2, 3) -> 1.0
(2, 4) -> 1.0
(2, 5) -> 1.0
(3, 1) -> -0.6846934429310386
(3, 2) -> -0.6846934429434928
(3, 3) -> -0.6846934429309881
(3, 4) -> 1.0
(3, 5) -> 1.0
(4, 1) -> -0.9080762312281941
(4, 2) -> -0.9080762353782766
(4, 3) -> -0.9080762312302715
(4, 4) -> -0.9087768720650484
(4, 5) -> 1.0
(5, 1) -> -0.9133201580817983
(5, 2) -> -0.9133201580814753
(5, 3) -> -0.9133201607262518
(5, 4) -> -0.9133201583647842
(5, 5) -> -1.0
